![Nuclio logo](https://nuclio.school/wp-content/uploads/2018/12/nucleoDS-newBlack.png)

# Proyecto final - Data Analytics y Business Intelligence

Recibimos dos datasets:

1. `renfe.csv`: Información de búsquedas de billetes que se hicieron en la página de Renfe.
2. `coordenadas_ciudades.csv`: Latitud y longitud de provincias españolas.

Queremos usar estos datasets para un modelo de Machine Learning que utilizaremos para predecir los precios de los billetes. Y, para ello, necesitamos limpiar, explorar y pre-procesar el dataset.

## Reglas de juego

1. El proyecto se debe entregar en grupos de dos o individualmente. 
2. Cada respuesta correcta suma un punto.
3. La calificación final consistirá en la suma de todos los puntos obtenidos sobre el total de puntos posibles.


## Diccionario de datos

Esta es la información provista:

### `renfe.csv`
- `FECHA_CONSULTA`: Fecha en la que se consultó la página.
- `FECHA_INICIO`: Fecha de inicio del trayecto.
- `FECHA_FIN`: Fecha de finalización del trayecto.
- `CIUDAD_ORIGEN`: Ciudad de origen del trayecto.
- `CIUDAD_DESTINO`: Ciudad destino del trayecto.
- `TIPO_TREN`: Tipo de tren.
- `TIPO_TARIFA`: Tipo de tarifa del billete.
- `CLASE`: Clase del asiento seleccionado.
- `PRECIO`: Precio del tren seleccionado.

### `coordenadas_ciudades.csv`
- `ciudad`: Nombre de la ciudad.
- `latitud`: Coordenada de latitud de la ciudad.
- `longitud`: Coordenada de longitud de la ciudad.

## Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import plotly as plt
import matplotlib
import plotly.express as px

import plotly.io as pio
pio.templates.default = "plotly_white"

pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("plotting.backend", "plotly")

## P0: Lee el dataset `renfe.csv`

In [ ]:
df = pd.read_csv("../data/renfe.csv", sep = ";")
df

## P1: Visualiza las primeras y las últimas filas del dataset

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

## P2: ¿Cuantas filas y columnas tiene el dataset?

In [ ]:
df.shape

### Tiene 383568 filas, y 9 columnas

## P3: Cambia los nombres de todas las columnas a minúsculas

In [ ]:
df.columns = df.columns.str.lower()
df

## P4: Muestra los tipos de datos de cada columna

In [ ]:
df.info()

## Exceptuando la columna "PRECIO" todas son de clase object

## P5: Cambia los tipos de datos que creas que creas incorrectos, por los tipos adecuados

In [ ]:
columnas_string = ["ciudad_origen", "ciudad_destino", "tipo_tren", "tipo_tarifa", "clase"]
df[columnas_string] = df[columnas_string].astype("string")

In [ ]:
df.info()

## P6: Filas duplicadas

In [ ]:
df.duplicated().value_counts()

### P6.1: ¿Cuántas filas duplicadas tiene el dataset?

In [ ]:
df.duplicated().sum()
## hay 33 filas que se repeten en el dataset

### P6.2: Quita las filas duplicadas

In [ ]:
df = df.drop_duplicates()
df

## P7: Valores nulos y análisis de `precio`

In [ ]:
df.describe()

### P7.1: ¿Que porcentaje de valores nulos hay por cada columna?

In [ ]:
df.isnull().value_counts()

### P7.2: ¿Cual es el mínimo, percentiles importantes (25%, 50%, 75%) y el máximo de `precio`?

In [ ]:
df.describe()

### P7.3: ¿Hay algo raro en el valor mínimo de `precio`? Quita las filas con ese valor del dataset

In [ ]:
df = df[df["precio"] != 0]
df

### P7.4: Reemplaza los valores nulos en `precio` por la media de esa columna

In [ ]:
df["precio"].mean()

In [ ]:
df["precio"] = df["precio"].fillna(df["precio"].mean())
df

In [ ]:
df[df["precio"].isnull()]

## Compruebo que no haya ningun NaN 

### P7.5: Quita las filas donde `clase` o `tipo_tarifa` sean nulos

In [ ]:
df.clase.isnull().value_counts()

In [ ]:
df.tipo_tarifa.isnull().value_counts()

In [ ]:
## Veo que tienen la misma canitdad de nulos, voy a ver si son las mismas filas:

assert sorted(df.clase.isnull()) == sorted(df.tipo_tarifa.isnull())

In [ ]:
## Confirmamos las sospechas: son las mismas filas. Procedo a eliminarlo.

df = df.drop(df.query("clase.isnull()").index).reset_index(drop=True)

In [ ]:
## compruebo que tanto los nulos de las columnas "clase" y "tipo_tarifa" se hayan eliminado correctamente

df.clase.isnull().value_counts()
df.tipo_tarifa.isnull().value_counts()

## P8: Tiempo de viaje

### P8.1: Calcula el tiempo de viaje en minutos (fecha_fin - fecha_inicio)

In [ ]:
## Calculo la duracion del viaje, debo pasar a datetime las colimnas "fecha_inicio" y "fecha_fin"

duracion = pd.to_datetime(df["fecha_fin"]) - pd.to_datetime(df["fecha_inicio"])
duracion

In [ ]:
## Creo una nueva columna con la duracion

df["tiempo_de_viaje"] = pd.to_timedelta(duracion)
df

### P8.2: Haz un histograma de la variable que acabas de crear (`tiempo_de_viaje`)

In [ ]:
## Cambio la duracion a minutos para que sea mas entendible en el grafico,

duracion_min = df["tiempo_de_viaje"].dt.total_seconds()/60

histograma = px.histogram(df, x=duracion_min, nbins = 100, title= "Numero de viajes segun duración", labels={"x": "Duración del viaje (minutos)"})

histograma.update_yaxes(title_text="Número de viajes")

histograma.show()

In [ ]:
## El histograma nos muestra que la mayoria de los viajes duran entre 1H30' y 3H

## P9: Extrae el día, el nombre del día, el mes y la hora de `fecha_inicio`

In [ ]:
df['nombre_dia_inicio'] = pd.to_datetime(df['fecha_inicio']).dt.day_name(locale= "es_ES")
df['dia_inicio'] = pd.to_datetime(df['fecha_inicio']).dt.day
df['nombre_mes_inicio'] = pd.to_datetime(df['fecha_inicio']).dt.month_name(locale= "es_ES")
df['hora_inicio'] = pd.to_datetime(df['fecha_inicio']).dt.time

## Me quedo con el numero del mes para la correlacion de mas adelante
num_mes = pd.to_datetime(df['fecha_inicio']).dt.month

In [ ]:
df

## P10: Quita las columnas `fecha_consulta`, `fecha_inicio` y `fecha_fin` del dataset

In [ ]:
df = df.drop(columns= ["fecha_consulta","fecha_inicio", "fecha_fin"]).reset_index(drop= True)

## P11: Lee el dataset `coordenadas_ciudades.csv` y únelo con al dataset que has procesado hasta ahora (utiliza `ciudad_destino` para el `join`)

In [ ]:
coordenadas_df = pd.read_csv('./coordenadas_ciudades.csv').set_index("ciudad")
coordenadas_df

In [ ]:
df = df.join(coordenadas_df, on='ciudad_destino', how= 'left', validate= "m:1")
df

In [ ]:
## Reordeno las columnas para que latitud y longitud queden despues de ciudad_destino

df = df.reindex(columns=['ciudad_origen', 'ciudad_destino', 'latitud', 'longitud', 'tipo_tren', 'tipo_tarifa', 'clase', 'precio', 'tiempo_de_viaje', 'nombre_dia_inicio', 'dia_inicio', 'nombre_mes_inicio', 'hora_inicio'])
df

## P12: Gráfica en un mapa el precio medio por ciudad de destino

In [ ]:
## Hago una serie con los precios medios para cada ciudad destino

precios_medios = df.groupby("ciudad_destino").precio.mean()
precios_medios

In [ ]:
## Creo un nuevo dataset con los datos que me interesan para el mapa: precio medio, ciudad y sus coordenadas.

coor_precios_df = coordenadas_df.join(precios_medios, on= "ciudad").reset_index()
coor_precios_df

In [ ]:
fig = px.scatter_map(coor_precios_df, lat= "latitud", lon="longitud", size= "precio", title= "Precio medio por Ciudad", hover_name= "ciudad", zoom= 4)

fig.update_layout(margin=dict(l=50, r=50, t=50, b=50), height=500, width=500)

fig.show()

## P13: Haz una tabla de correlación, ¿hay variables númericas correladas con el precio?

In [ ]:
## añado todas las variables numericas que puedan tener algun tipo de correlacion, obviando la longitud y latitud

tabla_correlacion = df[["precio","dia_inicio"]].join(duracion_min).join(num_mes).corr()

## rescato el mes en forma numerica para este punto y le cambio el nombre a la columna y fila para que sea mas entendible para todo el mundo
tabla_correlacion.rename(index= {"fecha_inicio":"num_mes"},columns= {"fecha_inicio":"num_mes"})

In [ ]:
## de esta tabla de correlacion podemos ver que el precio tiene una correlacion negativa con el tiempo de viaje y con el numero del mes: a mayor tiempo de viaje menor precio, y a mayor mes, menor precio... La verdad, intuia que iba ser al reves.

## P14: Relación entre variables del dataset y `precio`

### P14.1: Haz un scatter plot de precio vs. tiempo de viaje

In [ ]:
fig = px.scatter(df, x= duracion_min, y= "precio", hover_data="ciudad_destino", title= "Precio vs. tiempo de viaje")

fig.update_xaxes(title_text= "Duracion del viaje").update_yaxes(title_text= "Precio")

fig.show()

### P14.2: Haz un boxplot de precio vs. dia de la semana

In [ ]:
fig = px.box(df, x= "nombre_dia_inicio", y= "precio", hover_data="ciudad_destino", title= "Precio vs. dia de la semana", width= 750, height=1000) #, points="all")

fig.update_xaxes(title_text= "Dia de la semana", categoryarray=["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]).update_yaxes(title_text= "Precio")

fig.show()

### P14.3: Gráfica el precio medio por día de la semana

In [ ]:
## Ordeno los dias de la semana para luego el grafico

df["nombre_dia_inicio"] = pd.Categorical(df.nombre_dia_inicio, categories=["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"], ordered= True)

In [ ]:
df.groupby("nombre_dia_inicio").precio.mean()

In [ ]:
fig = px.line(df.groupby("nombre_dia_inicio").precio.mean(), y= "precio", title= "Precio medio por dia de la semana")

fig.update_xaxes(title_text= "Dia de la semana").update_yaxes(title_text= "Precio")

fig.show()

## P15: Crea un nuevo dataframe donge apliques *one-hot-encoding* a las variables categoricas

In [ ]:
df_dummies = pd.get_dummies(df[['ciudad_origen', 'ciudad_destino','tipo_tren', 'tipo_tarifa', 'clase', 'nombre_dia_inicio', 'nombre_mes_inicio']], dtype=int)
df_dummies